In [2]:
import os
import numpy as np
from ase import io
from ase.atoms import Atoms

In [3]:
def make_strained_rattled_ensembles(
    atoms_or_filename,
    scalings,
    n_per_scaling=5,
    rattle_stdev=0.01,
    epsmax=0.05,
    seed=None,
    out_dir=None,
    vasp_direct=True,
    vasp_sort=False,
):
    """
    Generate ensembles with:
      1) hydrostatic scaling
      2) additional random symmetric strain/shear
      3) atomic rattling

    Each generated structure is written to:
        out_dir/sXXXX_kYYY/POSCAR
    """

    rng = np.random.default_rng(seed)

    # Load structure
    if isinstance(atoms_or_filename, Atoms):
        base = atoms_or_filename.copy()
    else:
        base = io.read(atoms_or_filename)

    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)

    generated = []

    for s in scalings:
        for k in range(n_per_scaling):

            atoms = base.copy()

            # (1) Hydrostatic scaling
            atoms.set_cell(atoms.cell * float(s), scale_atoms=True)

            # (2) Random symmetric strain/shear
            E = np.zeros((3, 3))
            for i in range(3):
                for j in range(i, 3):
                    val = rng.uniform(-epsmax, epsmax)
                    E[i, j] = val
                    E[j, i] = val

            D = np.eye(3) + E

            cell = atoms.get_cell().array
            new_cell = (D @ cell.T).T
            atoms.set_cell(new_cell, scale_atoms=True)

            # (3) Rattle
            atoms.rattle(
                stdev=rattle_stdev,
                seed=int(rng.integers(0, 2**31 - 1))
            )

            generated.append(atoms)

            # Write structure
            if out_dir is not None:
                folder_name = f"s{int(round(s*1000)):04d}_k{k:03d}"
                folder_path = os.path.join(out_dir, folder_name)
                os.makedirs(folder_path, exist_ok=True)

                io.write(
                    os.path.join(folder_path, "POSCAR"),
                    atoms,
                    format="vasp",
                    direct=vasp_direct,
                    sort=vasp_sort
                )

    return generated

In [4]:
os.listdir()

['NiO',
 'Li2O2',
 'LiNi2O4',
 'Li2O',
 'Li2NiO3',
 'LiNi2O4.vasp',
 'Li2O2.vasp',
 'Li2O.vasp',
 'Li2NiO3.vasp',
 'rattling.ipynb',
 'Li2NiO3_2x1x2.vasp',
 'NiO.vasp',
 'NiO_2x2x2.vasp',
 'Li2O2_2x2x1.vasp',
 'Li2O_2x1x1.vasp',
 'Ni3O4.vasp',
 'Ni3O4',
 'MP',
 'Li3Ni',
 'Li6NiO4_P4_2-2_1-2',
 'Li6NiO4_Pmmn',
 'LiNi3',
 'LiO2',
 'LiO8',
 'Ni5O6',
 'Ni6O7',
 'LiNi3O4',
 'Li2NiO2',
 'LiNi9O13',
 'LiNi9O10',
 'Li7Ni5O12',
 'LiNi2O3',
 'LiNi4O5',
 'LiNiO3']

In [6]:
atoms = io.read("LiNiO3/LiNiO3.poscar")
supercell = atoms.repeat((2, 2, 2))         # careful!

io.write("LiNiO3/LiNiO3_2x2x2.vasp", supercell, format="vasp", direct=True, sort=True)

In [9]:
scalings = [0.95, 0.96, 0.97, 0.98, 0.99,
            1.00,
            1.01, 1.02, 1.03, 1.04, 1.05]

structures = make_strained_rattled_ensembles(
    atoms_or_filename="/nfshome/sicolo/work/MLIP/structure_prep/defective_structures/antiphase/POSCAR",
    scalings=scalings,
    n_per_scaling=2,
    rattle_stdev=0.01,
    epsmax=0.05,
    seed=42,                      # optional but good for reproducibility
    out_dir="/nfshome/sicolo/work/MLIP/structure_prep/defective_structures/antiphase/."
)